# Pseudobulk eQTL phenotype aggregation, QC and normalization

## Description

Aggregate per-nucleus Seurat counts into per-sample pseudobulk matrices, then QC and normalize to QTL-ready expression.

Per sample the pipeline counts cells, aggregates raw counts by `projid`, drops samples with fewer than 10 cells, applies `filterByExpr`, TMM-normalizes, voom-transforms to log2CPM, removes genes with mean log2CPM below 2.0, and quantile-normalizes.

Three aggregation entry points are provided:

- `seuratagg` for one Seurat object per cell type (listed in a celltype index file).
- `subtypeagg` for a combined object holding multiple cell types/subtypes (subset by `predicted.id`).
- `neuronsagg` for a cell type split across multiple Seurat objects (merged by `projid`).

**Timing:** ~5-10 min on typical compute infrastructure.

## Input Files

The toy example uses single-nuclei RNA-seq Seurat objects under `input/snrnaseq/`. The `MIC` (microglia) object backs `seuratagg`/`subtypeagg`; three toy `neuron` set objects back `neuronsagg`.

| File | Description |
|------|-------------|
| `protocol_example.snrnaseq.celltype.txt` | Tab-separated index: cell-type name + Seurat rds path (input to `seuratagg`) |
| `protocol_example.snrnaseq.seurat_MIC_projid.rds` | Seurat object (MIC cells) with a `projid` column in `meta.data` |
| `protocol_example.snrnaseq.subtype.txt` | Tab-separated index: subtype name + Seurat rds path (input to `subtypeagg`) |
| `protocol_example.snrnaseq.seurat_MIC_subtype.rds` | Seurat object with a `predicted.id` column holding two toy MIC subtypes |
| `protocol_example.snrnaseq.neuron.txt` | Tab-separated index: tissue label (`neuron`) + Seurat rds path (input to `neuronsagg`) |
| `protocol_example.snrnaseq.seurat_neuron_set1/2/3.rds` | Three Seurat objects, one cell type split across files, each with a `projid` column (merged by `projid` in `neuronsagg`) |

## Output Files

Written to `{cwd}/{name}.{celltype}.normalized.log2cpm.tsv` (e.g. `output/snrna_seq/aggregation/protocol_example.MIC.normalized.log2cpm.tsv`; `subtypeagg` writes one per subtype, e.g. `protocol_example.MIC_activated.normalized.log2cpm.tsv`; `neuronsagg` writes one for the merged cell type, e.g. `protocol_example.neuron.normalized.log2cpm.tsv`).

| File | Description |
|------|-------------|
| `{name}.{celltype}.normalized.log2cpm.tsv` | Quantile-normalized log2CPM matrix: `id` (gene) + one column per sample (`projid`) |

## Steps

`seuratagg`, `subtypeagg`, and `neuronsagg` are all runnable on the included toy data.

### seuratagg

Aggregate one Seurat object per cell type (listed in the index file) into a normalized log2CPM matrix.

In [ ]:
sos run pipeline/pseudobulk_expression_aggregation_QC_norm.ipynb seuratagg \
    --name protocol_example \
    --seurat-rds input/snrnaseq/protocol_example.snrnaseq.celltype.txt \
    --cwd output/snrna_seq/aggregation 


### subtypeagg

Subset a combined Seurat object by `predicted.id` (the subtype/cell-type column), then aggregate per subtype. The toy object splits MIC into two example subtypes, so this runs end-to-end.

In [ ]:
sos run pipeline/pseudobulk_expression_aggregation_QC_norm.ipynb subtypeagg \
    --name protocol_example \
    --seurat-rds input/snrnaseq/protocol_example.snrnaseq.subtype.txt \
    --cwd output/snrna_seq/aggregation


### neuronsagg

Merge a cell type that is split across multiple Seurat objects (here three toy `neuron` set files) by `projid`, then aggregate into a normalized log2CPM matrix. Counts for samples shared across sets are summed; samples unique to a set are appended.

In [ ]:
sos run pipeline/pseudobulk_expression_aggregation_QC_norm.ipynb neuronsagg \
    --name protocol_example \
    --seurat-rds input/snrnaseq/protocol_example.snrnaseq.neuron.txt \
    --cwd output/snrna_seq/aggregation 


## Command interface

In [ ]:
sos run pipeline/pseudobulk_expression_aggregation_QC_norm.ipynb -h

## Workflow implementation

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
# It is required to input the name of the analysis
parameter: name = str
parameter: cwd = path('output')
parameter: container = ""
# For cluster jobs, number commands to run per job
parameter: job_size = 5
# Wall clock time expected
parameter: walltime = "20h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 2

In [ ]:
[seuratagg]
import pandas as pd
# load seurat_rds rds output file
parameter: seurat_rds = path()


#for each tissue.
rds_result = pd.read_csv(seurat_rds, sep = "\t", header=None)
input_inv = rds_result.values.tolist()
tissue_id_inv = [x[0] for x in input_inv]
file_inv = [x[1] for x in input_inv]

input: file_inv, group_by = 1, group_with = "tissue_id_inv"
#output: gene_expression_matrix = f'{cwd:a}/{name}.{_tissue_id_inv}.count_matrix'
#output: cell_counts = f'{cwd:a}/{name}.{_tissue_id_inv}.nCells'
output: normalized_log2cpm = f'{cwd:a}/{name}.{_tissue_id_inv}.normalized.log2cpm.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
library(Seurat)
library(edgeR)
library(limma)

#loading separate seurat objects, each of a specific celltype
seu = readRDS(${_input:r})

#keep cell counts for sample filtering, (sample must be in metadata under 'sample')
cellcounts=table(seu@meta.data$projid)
# Convert table to data frame
cellcounts_df <- as.data.frame(cellcounts)
# Rename columns
names(cellcounts_df) <- c("sample", "ncell")
# cell counts of each sample output if you need it
#write.table(cellcounts_df, file = "${_output['cell_counts']}", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

seu=SetIdent(seu,value="projid")

#creation of the raw count pseudobulk
expr=AggregateExpression(seu,group.by="projid",slot="counts")$RNA

# delete the g prefix of colname: only for projids version.
colnames(expr) <- gsub("^g", "", colnames(expr))
colnames(expr) <- gsub("-", "_", colnames(expr))  # restore underscores Seurat replaced with dashes during AggregateExpression (keeps SAMPLE_001 IDs)
# raw counts matrix output if you need it
#write.table(expr, file = "${_output['gene_expression_matrix']}", sep = "\t", row.names = TRUE, quote = FALSE, col.names = TRUE)

#filtering out samples with fewer than 10 cells in a celltype
sampnames=names(cellcounts[cellcounts>9])
expr=expr[,sampnames]


#filter low expression genes
y <- DGEList(counts = expr)
keep <- filterByExpr(y)
y <- y[keep,,keep.lib.sizes=F]


#counts per million
y <- calcNormFactors(y, method = "TMM")
v <- voom(y, plot=F)
logcpm <- v$E

# remove genes if mean log2CPM < 2.0
mean_logcpm <- apply(logcpm, 1, mean)
logcpm <- logcpm[mean_logcpm > 2.0,]

logcpm <- as.data.frame(logcpm)
logcpm$id <- rownames(logcpm)
rownames(logcpm) <- NULL  #the rownames are now in the id column
logcpm <- logcpm[, c("id", setdiff(names(logcpm), "id"))]

# convert log2CPM to matrix
logcpm_id <- logcpm$id
logcpm <- as.matrix(logcpm[, colnames(logcpm) != "id"])
rownames(logcpm) <- logcpm_id

# quantile normalizarion
logcpm <- t(apply(logcpm, 1, rank, ties.method = "average"))
logcpm <- qnorm(logcpm / (ncol(logcpm) + 1))

# export
df <- data.frame(id = rownames(logcpm), logcpm, check.names = F)
write.table(df, file="${_output['normalized_log2cpm']}", sep="\t", quote = F, row.names = F)
cat("the normalized aggregated pseudo_bulk_eqtl tsv are saved")

In [ ]:
[subtypeagg]
import pandas as pd
# load seurat rds output file
parameter: seurat_rds = path()


#for each tissue.
rds_result = pd.read_csv(seurat_rds, sep = "\t", header=None)
input_inv = rds_result.values.tolist()
tissue_id_inv = [x[0] for x in input_inv]
file_inv = [x[1] for x in input_inv]

input: file_inv, group_by = 1, group_with = "tissue_id_inv"
#output: gene_expression_matrix = f'{cwd:a}/{name}.{_tissue_id_inv}.count_matrix' # raw count matrix output
#output: cell_counts = f'{cwd:a}/{name}.{_tissue_id_inv}.nCells' # cell counts of each sample output
output: normalized_log2cpm = f'{cwd:a}/{name}.{_tissue_id_inv}.normalized.log2cpm.tsv' # normalized log2cpm output
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
library(Seurat)
library(edgeR)
library(limma)

seu_all=readRDS(${_input:r})
seu_all=SetIdent(seu_all,value="predicted.id") #this is the subtype/celltype column name.
ct='${_tissue_id_inv}'
seu=subset(seu_all,idents=ct)

#keep cell counts for sample filtering, (sample must be in metadata under 'sample')
cellcounts=table(seu@meta.data$projid)

# Convert table to data frame
cellcounts_df <- as.data.frame(cellcounts)
# Rename columns
names(cellcounts_df) <- c("sample", "ncell")
# cell counts of each sample output if you need it
#write.table(cellcounts_df, file = "${_output['cell_counts']}", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

seu=SetIdent(seu,value="projid")

#creation of the raw count pseudobulk
expr=AggregateExpression(seu,group.by="projid",slot="counts")$RNA

# delete the g prefix of colname
colnames(expr) <- gsub("^g", "", colnames(expr))
colnames(expr) <- gsub("-", "_", colnames(expr))  # restore underscores Seurat replaced with dashes during AggregateExpression (keeps SAMPLE_001 IDs)
# raw counts matrix output if you need it
#write.table(expr, file = "${_output['gene_expression_matrix']}", sep = "\t", row.names = TRUE, quote = FALSE, col.names = TRUE)

#filtering out samples with fewer than 10 cells in a celltype
sampnames=names(cellcounts[cellcounts>9])
expr=expr[,sampnames]

#filter low expression genes
y <- DGEList(counts = expr)
keep <- filterByExpr(y)
y <- y[keep,,keep.lib.sizes=F]

#counts per million
y <- calcNormFactors(y, method = "TMM")

v <- voom(y, plot=F)
logcpm <- v$E

# remove genes if mean log2CPM < 2.0
mean_logcpm <- apply(logcpm, 1, mean)
logcpm <- logcpm[mean_logcpm > 2.0,]

logcpm <- as.data.frame(logcpm)
logcpm$id <- rownames(logcpm)
rownames(logcpm) <- NULL
logcpm <- logcpm[, c("id", setdiff(names(logcpm), "id"))]

# convert log2CPM to matrix
logcpm_id <- logcpm$id
logcpm <- as.matrix(logcpm[, colnames(logcpm) != "id"])
rownames(logcpm) <- logcpm_id

# quantile normalizarion
logcpm <- t(apply(logcpm, 1, rank, ties.method = "average"))
logcpm <- qnorm(logcpm / (ncol(logcpm) + 1))

# export
df <- data.frame(id = rownames(logcpm), logcpm, check.names = F)
write.table(df, file="${_output['normalized_log2cpm']}", sep="\t", quote = F, row.names = F)

cat("the normalized aggregated pseudo_bulk_eqtl tsv are saved")

In [ ]:
[neuronsagg]
import pandas as pd
# load seurat_rds index (tissue label + rds path); needed to define _tissue_id_inv for output naming
parameter: seurat_rds = path()
rds_result = pd.read_csv(seurat_rds, sep = "\t", header=None)
input_inv = rds_result.values.tolist()
tissue_id_inv = [x[0] for x in input_inv]
file_inv = [x[1] for x in input_inv]

input: file_inv, group_by = 1, group_with = "tissue_id_inv"
#output: gene_expression_matrix = f'{cwd:a}/{name}.{_tissue_id_inv}.count_matrix'
#output: cell_counts = f'{cwd:a}/{name}.{_tissue_id_inv}.nCells'
output: normalized_log2cpm = f'{cwd:a}/{name}.{_tissue_id_inv}.normalized.log2cpm.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
library(Seurat)
library(edgeR)
library(limma)
library(dplyr)

# Neurons were split into three files, so handled separately
seu1 = readRDS("input/snrnaseq/protocol_example.snrnaseq.seurat_neuron_set1.rds")
cellcounts1 = table(seu1@meta.data$projid)
seu1 = SetIdent(seu1, value = "projid")
expr1 = AggregateExpression(seu1, group.by = "projid", slot = "counts")$RNA

seu2 = readRDS("input/snrnaseq/protocol_example.snrnaseq.seurat_neuron_set2.rds")
cellcounts2 = table(seu2@meta.data$projid)
seu2 = SetIdent(seu2, value = "projid")
expr2 = AggregateExpression(seu2, group.by = "projid", slot = "counts")$RNA

seu3 = readRDS("input/snrnaseq/protocol_example.snrnaseq.seurat_neuron_set3.rds")
cellcounts3 = table(seu3@meta.data$projid)
seu3 = SetIdent(seu3, value = "projid")
expr3 = AggregateExpression(seu3, group.by = "projid", slot = "counts")$RNA


ct = "neuron"
genes1 = rownames(expr1)
genes2 = rownames(expr2)
genes3 = rownames(expr3)

# Find common genes among all three sets
common_genes = Reduce(intersect, list(genes1, genes2, genes3))

# Filter the expression matrices to keep only common genes
expr1 = expr1[common_genes, ]
expr2 = expr2[common_genes, ]
expr3 = expr3[common_genes, ]


# Combine cell counts from all three sets
df_cellcounts1 <- as.data.frame.table(cellcounts1)
df_cellcounts2 <- as.data.frame.table(cellcounts2)
df_cellcounts3 <- as.data.frame.table(cellcounts3)
colnames(df_cellcounts1) <- c("projid", "ncell")
colnames(df_cellcounts2) <- c("projid", "ncell")
colnames(df_cellcounts3) <- c("projid", "ncell")


# merge by projid, get the sum of cellcounts
cellcounts <- bind_rows(df_cellcounts1, df_cellcounts2, df_cellcounts3) %>%
  group_by(projid) %>%
  summarise(ncell = sum(ncell, na.rm = TRUE))
cat("merge by projid, get the sum of cellcounts:\n")
print(cellcounts)
#write.table(cellcounts, file = "${_output['cell_counts']}", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)

#filtering out samples with fewer than 10 cells in a celltype
#sampnames=names(cellcounts[cellcounts>9])
filtered_cellcounts = cellcounts %>% filter(ncell > 9)
sampnames = filtered_cellcounts$projid

# commen samples of expr1 and expr2
expr2 = expr2[rownames(expr1), ]
shared12 = colnames(expr1)[colnames(expr1) %in% colnames(expr2)]

# for each overlapped sample, add expr2 on expr1
if(length(shared12) > 0) {
  expr1[, shared12] = expr1[, shared12] + expr2[, shared12]
}

# remove overlapped samples in expr2
expr2 = expr2[, !colnames(expr2) %in% shared12]
combined12 = cbind(expr1, expr2)

expr3 = expr3[rownames(combined12), ]
shared123 = colnames(combined12)[colnames(combined12) %in% colnames(expr3)]
if(length(shared123) > 0) {
  combined12[, shared123] = combined12[, shared123] + expr3[, shared123]
}
expr3 = expr3[, !colnames(expr3) %in% shared123]
expr = cbind(combined12, expr3)

# delete the g prefix of colname
colnames(expr) <- gsub("^g", "", colnames(expr))
colnames(expr) <- gsub("-", "_", colnames(expr))  # restore underscores Seurat replaced with dashes during AggregateExpression (keeps SAMPLE_001 IDs)
# raw counts matrix output if you need it
#write.table(expr, file = "${_output['gene_expression_matrix']}", sep = "\t", row.names = TRUE, quote = FALSE, col.names = TRUE)

expr=expr[,sampnames]
cat("Dimension of expr matrix after filtering:\n")
print(dim(expr))

#filter low expression genes
y <- DGEList(counts = expr)
keep <- filterByExpr(y)
y <- y[keep,,keep.lib.sizes=F]

#counts per million
y <- calcNormFactors(y, method = "TMM")
v <- voom(y, plot=F)
logcpm <- v$E

# remove genes if mean log2CPM < 2.0
mean_logcpm <- apply(logcpm, 1, mean)
logcpm <- logcpm[mean_logcpm > 2.0,]

logcpm <- as.data.frame(logcpm)
logcpm$id <- rownames(logcpm)
rownames(logcpm) <- NULL
logcpm <- logcpm[, c("id", setdiff(names(logcpm), "id"))]

# convert log2CPM to matrix
logcpm_id <- logcpm$id
logcpm <- as.matrix(logcpm[, colnames(logcpm) != "id"])
rownames(logcpm) <- logcpm_id

# quantile normalizarion
logcpm <- t(apply(logcpm, 1, rank, ties.method = "average"))
logcpm <- qnorm(logcpm / (ncol(logcpm) + 1))

# export
df <- data.frame(id = rownames(logcpm), logcpm, check.names = F)
write.table(df, file="${_output['normalized_log2cpm']}", sep="\t", quote = F, row.names = F)

cat("the normalized aggregated pseudo_bulk_eqtl tsv are saved")

In [ ]:
# Python code. The example uses projid insdead of sampleid, and the length of projid number should be filled into 8 to match 
the projid--sample list. So after aggregatiom, should processed the column name in the tsv file.
# the log2CPM tsv file: 1st col is id, the rest are projids.
# fill projid to 8
import pandas as pd

def pad_column_names(df, pad_length=8):
    new_cols = [df.columns[0]] + [col if len(col) == pad_length or not col.isdigit() else col.zfill(pad_length) for col in df.columns[1:]]
    df.columns = new_cols
    return df

file_path = '/The input path/phenodata_quantnorm_nofill0/snuc_pseudo_bulk.Ast.normalized.log2cpm.tsv'
df = pd.read_csv(file_path, sep='\t')

df = pad_column_names(df)

output_file_path = 'The output path/For your tsv data'
df.to_csv(output_file_path, sep='\t', index=False, quotechar='', quoting=3)
